## **Project 1 Data Cleaning & Prep**

In [3]:
import pandas as pd
import os

raw_data = "Dataset for Data Analytics.xlsx"

if os.path.exists(raw_data):
    # Use read_excel to cleanly parse the spreadsheet layers
    df = pd.read_excel(raw_data)
    print("✅ Excel environment successfully established!")
    print(f"📊 Dataset Shape: {df.shape[0]} transaction rows loaded.")
else:
    print(f"❌ Error: Could not find '{raw_data}' in your folder workspace.")
    print("Double check that the file name is spelled exactly right (including capitals).")

✅ Excel environment successfully established!
📊 Dataset Shape: 1200 transaction rows loaded.


In [ ]:
# Print out a summary of missing values across all columns
print("--- Missing Values Audit ---")
print(df.isnull().sum())

# Check for duplicate Order IDs to test our unique identifier gate status
print("\n--- Unique ID Verification Check ---")
print(f"Total Rows in Dataset: {len(df)}")
print(f"Unique OrderID Counts: {df['OrderID'].nunique()}")
print(f"Rogue Duplicate OrderID Rows: {df['OrderID'].duplicated().sum()}")

--- Missing Values Audit ---
OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

--- Unique ID Verification Check ---
Total Rows in Dataset: 1200
Unique OrderID Counts: 1200
Rogue Duplicate OrderID Rows: 0


In [5]:
# Defensive Text Layer Scrubbing (Clean Leading/Trailing Spaces)
string_columns = df.select_dtypes(include=['object']).columns
for col in string_columns:
    df[col] = df[col].astype(str).str.strip()

In [6]:
# Structural Imputation of Missing Column Blocks
df['CouponCode'] = df['CouponCode'].replace(['nan', 'NaN', 'None'], np.nan)
df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')

In [7]:
# Mandatory Gate Check: Force Strict ISO 8601 Temporal Format (0% Format Error)
df['Date'] = pd.to_datetime(df['Date'], errors='raise')
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

In [8]:
# Mandatory Gate Check: Enforce Absolute Deduplication (0% Duplicate ID Error)
df = df.drop_duplicates(subset=['OrderID'], keep='first')

In [9]:
# Numeric Precision Engineering (Force Uniform 2-Decimal Precision)
df['UnitPrice'] = df['UnitPrice'].round(2)
df['TotalPrice'] = df['TotalPrice'].round(2)

In [10]:
# Final Sanity Check Assertions
assert df['OrderID'].duplicated().sum() == 0, "❌ Error Gate Violated: Duplicate unique IDs found!"
assert pd.to_datetime(df['Date'], errors='coerce').notnull().all(), "❌ Error Gate Violated: Bad date formatting!"

In [13]:
print("==================================================")
print("       🕵️‍♂️ PRE-EXPORT DATA INTEGRITY REPORT        ")
print("==================================================")

# 1. STRUCTURAL DIMENSION AUDIT
print(f"📊 Dataset Dimensions: {df.shape[0]} rows | {df.shape[1]} columns")
if df.shape[0] < 1000:
    print("⚠️ WARNING: Row count dropped significantly! Double-check your filters.")
else:
    print("✅ Row Count: Stable and safe.")

# 2. MANDATORY GATE 1: UNIQUE IDENTIFIER CHECK (0% Error Rate)
duplicate_ids = df['OrderID'].duplicated().sum()
print(f"\n🆔 Duplicate OrderIDs found: {duplicate_ids}")
if duplicate_ids == 0:
    print("✅ GATE 1 PASSED: Unique identifier error rate is exactly 0%.")
else:
    print("❌ GATE 1 FAILED: Duplicates still exist in primary keys!")

# 3. MANDATORY GATE 2: DATE COMPLIANCE CHECK (0% Error Rate)
# Coerce dates to see if any failed to format (they will turn into NaT / Null)
null_dates = pd.to_datetime(df['Date'], errors='coerce').isnull().sum()
print(f"\n📅 Malformed Date entries found: {null_dates}")
if null_dates == 0:
    print("✅ GATE 2 PASSED: All dates are perfectly uniform (ISO 8601).")
else:
    print("❌ GATE 2 FAILED: You have malformed date strings remaining!")

# 4. MISSING VALUES POST-AUDIT
print("\n🔍 Remaining Null/Missing Values Per Column:")
print(df.isnull().sum())

# 5. DATA TYPES & FORMAT PREVIEW
print("\n💾 Data Type Blueprint:")
print(df.dtypes)

print("\nVisual Verification (First 3 Records):")
display(df.head(3))

       🕵️‍♂️ PRE-EXPORT DATA INTEGRITY REPORT        
📊 Dataset Dimensions: 1200 rows | 14 columns
✅ Row Count: Stable and safe.

🆔 Duplicate OrderIDs found: 0
✅ GATE 1 PASSED: Unique identifier error rate is exactly 0%.

📅 Malformed Date entries found: 0
✅ GATE 2 PASSED: All dates are perfectly uniform (ISO 8601).

🔍 Remaining Null/Missing Values Per Column:
OrderID            0
Date               0
CustomerID         0
Product            0
Quantity           0
UnitPrice          0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
ItemsInCart        0
CouponCode         0
ReferralSource     0
TotalPrice         0
dtype: int64

💾 Data Type Blueprint:
OrderID             object
Date                object
CustomerID          object
Product             object
Quantity             int64
UnitPrice          float64
ShippingAddress     object
PaymentMethod       object
OrderStatus         object
TrackingNumber      object
ItemsInCart          int64
CouponCode

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.1
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.7
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.4


In [14]:
# Export the High-Integrity Golden Record
clean_output_path = "Clean_Dataset_Data_Analytics.csv"
df.to_csv(clean_output_path, index=False)

print("\n🚀 --- VERIFICATION GATE QUALITY REPORT ---")
print(f"✅ Total Verified Rows Saved: {len(df)}")
print(f"✅ Unique ID Error Rate: {df['OrderID'].duplicated().sum()}%")
print(f"✅ Missing Date Format Errors: {df['Date'].isnull().sum()}")
print(f"🎉 Clean File Output Generated Successfully: {clean_output_path}")


🚀 --- VERIFICATION GATE QUALITY REPORT ---
✅ Total Verified Rows Saved: 1200
✅ Unique ID Error Rate: 0%
✅ Missing Date Format Errors: 0
🎉 Clean File Output Generated Successfully: Clean_Dataset_Data_Analytics.csv
